In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
df = pd.read_csv("../data/diabetes_cdc.csv")

X = df.drop(columns="Diabetes_binary")  # the 21 features (inputs)
y = df["Diabetes_binary"]              # the target (what we predict)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size= 0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)
print(y_train.mean().round(3), y_test.mean().round(3))

(202944, 21) (50736, 21)
0.139 0.139


In [2]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score

baseline = DummyClassifier(strategy ="most_frequent")
baseline.fit(X_train, y_train)
base_pred = baseline.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, base_pred), 3))
print("Recall:", round(recall_score(y_test, base_pred), 3))


Accuracy: 0.861
Recall: 0.0


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import precision_score, f1_score, roc_auc_score

logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
logreg.fit(X_train, y_train)

lr_pred = logreg.predict(X_test)
lr_prob = logreg.predict_proba(X_test)[:, 1]

print("Accuracy: ", round(accuracy_score(y_test, lr_pred), 3))
print("Precision:", round(precision_score(y_test, lr_pred), 3))
print("Recall:   ", round(recall_score(y_test, lr_pred), 3))
print("F1:       ", round(f1_score(y_test, lr_pred), 3))
print("ROC-AUC:  ", round(roc_auc_score(y_test, lr_prob), 3))

Accuracy:  0.862
Precision: 0.517
Recall:    0.158
F1:        0.242
ROC-AUC:   0.819


Baseline vs. logistic regression:
- Baseline (always "no"): 86.1% accuracy but 0% recall, so it catches no diabetes cases.
- Logistic regression: 86.2% accuracy, precision 0.52, recall 0.16, F1 0.24, ROC-AUC 0.82.
- ROC-AUC of 0.82 shows the model ranks people by risk well, but the default 0.5 cutoff makes it miss 84% of diabetes cases.
- Next: use class_weight="balanced" and tune the decision threshold to raise recall.

In [4]:
logreg_bal = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter= 1000, class_weight="balanced")
)
logreg_bal.fit(X_train, y_train)

bal_pred = logreg_bal.predict(X_test)
bal_prob = logreg_bal.predict_proba(X_test)[:, 1]

print("Accuracy: ", round(accuracy_score(y_test, bal_pred), 3))
print("Precision:", round(precision_score(y_test, bal_pred), 3))
print("Recall:   ", round(recall_score(y_test, bal_pred), 3))
print("F1:       ", round(f1_score(y_test, bal_pred), 3))
print("ROC-AUC:  ", round(roc_auc_score(y_test, bal_prob), 3))

Accuracy:  0.732
Precision: 0.311
Recall:    0.761
F1:        0.441
ROC-AUC:   0.82


In [5]:
import pandas as pd

rows = []
for t in [0.3,0.4,0.5,0.6,0.7]:
    pred_t = (bal_prob >= t).astype(int)
    rows.append({
        "threshold": t,
        "precision": round(precision_score(y_test, pred_t),3),
        "recall": round(recall_score(y_test, pred_t),3),
        "f1": round(f1_score(y_test, pred_t),3),
        "flagged_%": round(pred_t.mean()*100, 1),
    })
pd.DataFrame(rows)

,threshold,precision,recall,f1,flagged_%
0,0.3,0.238,0.917,0.378,53.6
1,0.4,0.274,0.854,0.415,43.4
2,0.5,0.311,0.761,0.441,34.1
3,0.6,0.354,0.649,0.459,25.5
4,0.7,0.400,0.498,0.444,17.3


Balanced logistic regression + threshold tuning:
- class_weight="balanced" raised recall from 0.16 to 0.76 (F1 0.24 → 0.44), with ROC-AUC unchanged at 0.82. Balancing mostly shifts probabilities upward, which is similar to lowering the cutoff.
- Thresholds trade recall against precision: 0.3 catches 92% of cases but flags 54% of people; 0.7 flags 17% but catches only 50%.
- Best F1 is at threshold 0.6 (0.459). For screening, 0.5–0.6 is a reasonable balance; the right choice depends on clinic capacity and the cost of missed cases.
- Caveat: threshold was chosen on the test set; it should be tuned on a validation set (to do with cross-validation).

In [6]:
from sklearn.ensemble import RandomForestClassifier

rf= RandomForestClassifier(
    n_estimators = 100,
    min_samples_leaf= 50,
    class_weight = "balanced",
    n_jobs = -1,
    random_state = 42
)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

print("Accuracy: ", round(accuracy_score(y_test, rf_pred), 3))
print("Precision:", round(precision_score(y_test, rf_pred), 3))
print("Recall:   ", round(recall_score(y_test, rf_pred), 3))
print("F1:       ", round(f1_score(y_test, rf_pred), 3))
print("ROC-AUC:  ", round(roc_auc_score(y_test, rf_prob), 3))


Accuracy:  0.722
Precision: 0.306
Recall:    0.785
F1:        0.44
ROC-AUC:   0.824
